In [ ]:
!pip install xgboost

In [ ]:
#the imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score


df = pd.read_csv("/content/Final_Augmented_dataset_Diseases_and_Symptoms.csv")

df.head(10)

,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,stuttering or stammering,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,panic disorder,0,0,1,1,0,0,1,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,panic disorder,1,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,panic disorder,0,0,0,1,0,0,1,1,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,panic disorder,1,0,0,1,0,1,1,0,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,panic disorder,1,1,1,0,0,0,0,1,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
#removing rare diseases to make accuracy score more reliable
counts = df["diseases"].value_counts()
keep = counts[counts>100].index
filtered_df = df[df['diseases'].isin(keep)]
df = filtered_df


#spliting data into diseases and symptoms
X = df.drop('diseases', axis=1)
y = df['diseases']

#making sure all dieases are encoded to numbers
le = LabelEncoder()
y = le.fit_transform(y)

#splitting data to training and testing data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle = True, random_state=42, stratify = y
)



In [ ]:
#XGBoost ML model
from xgboost import XGBClassifier

#hyperparameters of XGBoost
xgb = XGBClassifier(
    objective='multi:softmax',
    num_class=len(np.unique(y)),
    eval_metric='merror',
    learning_rate=0.08,
    max_depth=3,
    n_estimators=200,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    early_stopping_rounds=20,
    tree_method="hist"
)


#training XGBoost
xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=True            # print progress
)


#XGBoost making predictions
xgb_pred = xgb.predict(X_test)

print("XGBoost Accuracy:", accuracy_score(y_test, xgb_pred))

In [ ]:
#extention robutness test 1- noise
#noise is more damaging because its wrong info

def an(X_test, noise_amount =0.1):
  noise_data = X_test.copy()
  noise_data = noise_data.to_numpy()
  x_patn, x_symn = noise_data.shape
  for z in range(x_patn):
    noise_add = np.random.choice(x_symn, size = int(noise_amount * x_symn), replace = True)
    noise_data[z, noise_add] = 1 - noise_data[z, noise_add] #this flips the value of some of the symptoms, showing user mistakes at times
  return noise_data

noise_test_data = an(X_test, noise_amount = 0.1)

In [ ]:
y_pred = xgb.predict(noise_test_data)
# preds_class = np.argmax(y_pred, axis=1)

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
#extention robustness test 2- missing data
#missing data is less damaging because it is only less info
import random

def rs(X_test, miss_rate = 0.1):
  miss_data = X_test.copy() #copies the testing dataset so we do not alter the regular one
  miss_data = miss_data.to_numpy() #cnverts to a numpy array
  x_pat, x_sym = miss_data.shape #spilts test data into num of rows and columns
  for z in range(x_pat): #loops through each patient/row
    data_gone = np.random.choice(x_sym, size = int(miss_rate * x_sym), replace = False)
    miss_data[z, data_gone] = -1 #sets a random percent of symptoms to false so they get "erased"
  return miss_data

missing_test_data = rs(X_test, miss_rate = 0.7)

In [ ]:
y_pred = xgb.predict(missing_test_data)

print("XGBoost Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
#making confusion matrix
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_test, xgb_pred)

plt.figure(figsize=(50,50))
sns.heatmap(cm, annot = True, fmt='d', cmap='Blues', xticklabels=False, yticklabels = False, cbar = True, annot_kws={'size':5}, linewidth=1)



In [ ]:
#making zoomed in confusion matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, xgb_pred)
short_cm = []

for i in range (0,10):
  short_cm.append(cm[i][0:10])

random_10 = (
    df["diseases"]
    .drop_duplicates()   # remove duplicates
    .sample(n=10, random_state=42)  # random selection
)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(font_scale=3)
plt.figure(figsize=(25,25))

sns.heatmap(short_cm, annot = True, fmt='d', cmap='Blues', xticklabels=random_10, yticklabels = random_10, cbar = True, annot_kws={'size':25}, linewidth=1)

plt.xlabel('predicted')
plt.ylabel('true')
plt.title('confusion matrix')

plt.xticks(rotation = 90)
plt.yticks(rotation=0)

plt.tight_layout()

plt.show()